In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [28]:
class FourIndexLayer(nn.Module):
    def __init__(self, N):
        super().__init__()
        # Complex weights and biases
        self.W = nn.Parameter(torch.randn(N, N, N, N, dtype=torch.complex64) * 0.1)
        self.b = nn.Parameter(torch.zeros(N, N, dtype=torch.complex64))

    def forward(self, x):
        # x: [batch, N, N] complex
        ys = []
        for xx in x:  # xx: [N, N] complex
            # y_{ab} = sum_{cd} W_{abcd} * xx_{cd} + b_{ab}
            y = torch.einsum('abcd,cd->ab', self.W, xx) + self.b
            ys.append(y)
        return torch.stack(ys)

In [29]:
class MatrixNet(nn.Module):
    def __init__(self, N, depth=2):
        super().__init__()
        self.layers = nn.ModuleList([FourIndexLayer(N) for _ in range(depth)])

    def forward(self, x):
        # x: [batch, N, N] complex
        for layer in self.layers:
            x = torch.tanh(layer(x))
        return x  # output: [batch, N, N] complex

In [30]:
# Define fixed A and B for the target function
N = 4 
A = torch.randn(N, N, dtype=torch.complex64)
B = torch.randn(N, N, dtype=torch.complex64)
A, B

(tensor([[ 1.1998-0.1574j,  0.6373-0.5554j, -0.7039-1.0864j,  0.3819-1.3114j],
         [ 0.0756-0.5395j, -0.0053-0.6152j, -0.8208-0.0869j, -1.9599+0.8681j],
         [-0.4195-0.1863j, -0.2006+0.1367j,  0.1942+0.3467j, -1.0001+1.1453j],
         [-0.3711-1.1124j,  0.1005+0.8200j, -0.4263-0.1863j,  1.6511+0.4498j]]),
 tensor([[-6.9685e-01-0.8846j,  7.2144e-01-0.5691j, -9.2764e-01-0.7205j,
           1.7476e-01+0.0385j],
         [ 4.9978e-01+1.1992j,  6.3921e-01-1.1646j,  2.7815e-04+0.1768j,
          -3.2700e-01+0.6001j],
         [ 8.3546e-01+0.7416j,  1.3680e+00-0.0796j,  1.0705e+00+0.2365j,
           1.6102e-01+0.2640j],
         [ 1.1970e+00+0.3054j,  1.4600e+00+1.0038j, -4.8450e-01+0.7547j,
           1.4272e-01-0.3385j]]))

In [31]:
# Generate training data
training_size = 100
X_train = []
Y_train = []
for _ in range(training_size):
    x = torch.randn(N, N, dtype=torch.complex64)
    y = torch.sin(torch.einsum('ij,jk->ik', A, x)) + B  # Y = sin(AX) + B
    X_train.append(x)
    Y_train.append(y)
X_train = torch.stack(X_train)
Y_train = torch.stack(Y_train)

In [32]:
# Instantiate and train the model
model = MatrixNet(N, depth=2)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
model

MatrixNet(
  (layers): ModuleList(
    (0-1): 2 x FourIndexLayer()
  )
)

In [33]:
for epoch in range(201):
    optimizer.zero_grad()
    Y_pred = model(X_train)
    loss = loss_fn(Y_pred.real, Y_train.real) + loss_fn(Y_pred.imag, Y_train.imag)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 73.42514038085938
Epoch 50, Loss: 71.9718246459961
Epoch 100, Loss: 70.50437927246094
Epoch 150, Loss: 69.7135009765625
Epoch 200, Loss: 69.4404067993164


In [35]:
# Example prediction
X_test = torch.randn(N, N, dtype=torch.complex64)
Y_test = torch.sin(torch.einsum('ij,jk->ik', A, X_test)) + B
Y_test_pred = model(X_test.unsqueeze(0))[0]
(Y_test_pred - Y_test)

tensor([[ 0.7403+1.3917e+00j, -0.1595-6.0557e-02j,  0.2038-2.6527e-01j,
          1.2080+3.3631e-02j],
        [ 5.1085+3.9645e+00j, -0.8260+8.1355e-01j,  0.6814-4.2583e-01j,
         -2.9697+1.1110e+00j],
        [ 1.4334-1.9257e+00j, -1.1051-1.7046e-03j, -0.2180+4.2272e-01j,
         -0.7622-3.9539e-01j],
        [ 0.3161-1.1541e+00j, -1.0223-4.8675e-01j, -0.4286-5.4435e+00j,
         -1.4481+1.4222e-01j]], grad_fn=<SubBackward0>)

In [ ]:
# Absolute values of differences
(Y_test_pred - Y_test).abs()

tensor([[1.5764, 0.1706, 0.3345, 1.2085],
        [6.4664, 1.1594, 0.8035, 3.1707],
        [2.4006, 1.1051, 0.4756, 0.8586],
        [1.1966, 1.1322, 5.4604, 1.4550]], grad_fn=<AbsBackward0>)